In [0]:
telemetry = spark.table("workspace.default.silver_telemetry")
failures = spark.table("workspace.default.silver_failures")

print("Telemetry Schema:")
telemetry.printSchema()

print("Failures Schema:")
failures.printSchema()

Telemetry Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

Failures Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

print("Total failure records:", failures.count())

print("\nFailure types:")
failures.groupBy("failure").count().orderBy("failure").show()

print("\nMachines with failures:", failures.select("machineID").distinct().count())

print("\nFailure date range:")
failures.select(
    col("datetime").alias("failure_datetime")
).agg(
    {"failure_datetime": "min"}
).show()

failures.select(
    col("datetime").alias("failure_datetime")
).agg(
    {"failure_datetime": "max"}
).show()

Total failure records: 761

Failure types:
+-------+-----+
|failure|count|
+-------+-----+
|  comp1|  192|
|  comp2|  259|
|  comp3|  131|
|  comp4|  179|
+-------+-----+


Machines with failures: 98

Failure date range:
+---------------------+
|min(failure_datetime)|
+---------------------+
|  2015-01-02 03:00:00|
+---------------------+

+---------------------+
|max(failure_datetime)|
+---------------------+
|  2015-12-31 06:00:00|
+---------------------+



In [0]:
from pyspark.sql.functions import col, when, lit, expr

telemetry_target = (
    telemetry.alias("t")
    .join(
        failures.alias("f"),
        (col("t.machineID") == col("f.machineID")) &
        (col("f.datetime") > col("t.datetime")) &
        (col("f.datetime") <= col("t.datetime") + expr("INTERVAL 24 HOURS")),
        "left"
    )
    .select(
        col("t.datetime").alias("datetime"),
        col("t.machineID").alias("machineID"),
        col("t.volt").alias("volt"),
        col("t.rotate").alias("rotate"),
        col("t.pressure").alias("pressure"),
        col("t.vibration").alias("vibration"),
        col("f.datetime").alias("failure_datetime")
    )
    .withColumn(
        "failure_next_24h",
        when(col("failure_datetime").isNotNull(), 1).otherwise(0)
    )
)

print("Telemetry records with 24-hour failure target:")
print(telemetry_target.count())

telemetry_target.groupBy("failure_next_24h").count().show()

Telemetry records with 24-hour failure target:
877126
+----------------+------+
|failure_next_24h| count|
+----------------+------+
|               0|858916|
|               1| 18210|
+----------------+------+



In [0]:
from pyspark.sql.functions import max

telemetry_target_clean = (
    telemetry_target
    .groupBy(
        "datetime",
        "machineID",
        "volt",
        "rotate",
        "pressure",
        "vibration"
    )
    .agg(
        max("failure_next_24h").alias("failure_next_24h")
    )
    .orderBy("machineID", "datetime")
)

print("Clean telemetry records:", telemetry_target_clean.count())

print("\nTarget distribution:")
telemetry_target_clean.groupBy(
    "failure_next_24h"
).count().show()

Clean telemetry records: 876100

Target distribution:
+----------------+------+
|failure_next_24h| count|
+----------------+------+
|               0|858916|
|               1| 17184|
+----------------+------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, stddev

window_7d = (
    Window
    .partitionBy("machineID")
    .orderBy(col("datetime").cast("long"))
    .rangeBetween(-7 * 24 * 60 * 60, 0)
)

features_7d = (
    telemetry_target_clean
    .withColumn("avg_volt_7d", avg("volt").over(window_7d))
    .withColumn("std_volt_7d", stddev("volt").over(window_7d))
    .withColumn("avg_rotate_7d", avg("rotate").over(window_7d))
    .withColumn("std_rotate_7d", stddev("rotate").over(window_7d))
    .withColumn("avg_pressure_7d", avg("pressure").over(window_7d))
    .withColumn("std_pressure_7d", stddev("pressure").over(window_7d))
    .withColumn("avg_vibration_7d", avg("vibration").over(window_7d))
    .withColumn("std_vibration_7d", stddev("vibration").over(window_7d))
)

print("7-day rolling features created.")

features_7d.printSchema()

features_7d.select(
    "datetime",
    "machineID",
    "avg_volt_7d",
    "std_volt_7d",
    "avg_rotate_7d",
    "std_rotate_7d",
    "avg_pressure_7d",
    "std_pressure_7d",
    "avg_vibration_7d",
    "std_vibration_7d",
    "failure_next_24h"
).show(10, truncate=False)

7-day rolling features created.
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- failure_next_24h: integer (nullable = true)
 |-- avg_volt_7d: double (nullable = true)
 |-- std_volt_7d: double (nullable = true)
 |-- avg_rotate_7d: double (nullable = true)
 |-- std_rotate_7d: double (nullable = true)
 |-- avg_pressure_7d: double (nullable = true)
 |-- std_pressure_7d: double (nullable = true)
 |-- avg_vibration_7d: double (nullable = true)
 |-- std_vibration_7d: double (nullable = true)

+-------------------+---------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+----------------+
|datetime           |machineID|avg_volt_7d       |std_volt_7d       |avg_rotate_7d     |std_rotate_7d     |

In [0]:
model_features = (
    features_7d
    .select(
        "datetime",
        "machineID",
        "avg_volt_7d",
        "std_volt_7d",
        "avg_rotate_7d",
        "std_rotate_7d",
        "avg_pressure_7d",
        "std_pressure_7d",
        "avg_vibration_7d",
        "std_vibration_7d",
        "failure_next_24h"
    )
    .dropna()
)

print("Model feature records:", model_features.count())

print("\nTarget distribution:")
model_features.groupBy(
    "failure_next_24h"
).count().show()

print("\nModel feature schema:")
model_features.printSchema()

Model feature records: 876000

Target distribution:
+----------------+------+
|failure_next_24h| count|
+----------------+------+
|               0|858835|
|               1| 17165|
+----------------+------+


Model feature schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- avg_volt_7d: double (nullable = true)
 |-- std_volt_7d: double (nullable = true)
 |-- avg_rotate_7d: double (nullable = true)
 |-- std_rotate_7d: double (nullable = true)
 |-- avg_pressure_7d: double (nullable = true)
 |-- std_pressure_7d: double (nullable = true)
 |-- avg_vibration_7d: double (nullable = true)
 |-- std_vibration_7d: double (nullable = true)
 |-- failure_next_24h: integer (nullable = true)



In [0]:
model_pd = model_features.toPandas()

print("Rows:", len(model_pd))
print("Machines:", model_pd["machineID"].nunique())

print("\nTarget distribution:")
print(model_pd["failure_next_24h"].value_counts())

print("\nFeature columns:")
print([
    "avg_volt_7d",
    "std_volt_7d",
    "avg_rotate_7d",
    "std_rotate_7d",
    "avg_pressure_7d",
    "std_pressure_7d",
    "avg_vibration_7d",
    "std_vibration_7d"
])

Rows: 876000
Machines: 100

Target distribution:
failure_next_24h
0    858835
1     17165
Name: count, dtype: int64

Feature columns:
['avg_volt_7d', 'std_volt_7d', 'avg_rotate_7d', 'std_rotate_7d', 'avg_pressure_7d', 'std_pressure_7d', 'avg_vibration_7d', 'std_vibration_7d']


In [0]:
model_pd = model_pd.sort_values(
    ["datetime", "machineID"]
).reset_index(drop=True)

split_index = int(len(model_pd) * 0.8)

train_pd = model_pd.iloc[:split_index].copy()
test_pd = model_pd.iloc[split_index:].copy()

print("Training records:", len(train_pd))
print("Testing records:", len(test_pd))

print("\nTraining period:")
print(train_pd["datetime"].min(), "to", train_pd["datetime"].max())

print("\nTesting period:")
print(test_pd["datetime"].min(), "to", test_pd["datetime"].max())

print("\nTraining target:")
print(train_pd["failure_next_24h"].value_counts())

print("\nTesting target:")
print(test_pd["failure_next_24h"].value_counts())

Training records: 700800
Testing records: 175200

Training period:
2015-01-01 07:00:00 to 2015-10-20 06:00:00

Testing period:
2015-10-20 07:00:00 to 2016-01-01 06:00:00

Training target:
failure_next_24h
0    686776
1     14024
Name: count, dtype: int64

Testing target:
failure_next_24h
0    172059
1      3141
Name: count, dtype: int64


In [0]:
feature_columns = [
    "avg_volt_7d",
    "std_volt_7d",
    "avg_rotate_7d",
    "std_rotate_7d",
    "avg_pressure_7d",
    "std_pressure_7d",
    "avg_vibration_7d",
    "std_vibration_7d"
]

X_train = train_pd[feature_columns]
y_train = train_pd["failure_next_24h"]

X_test = test_pd[feature_columns]
y_test = test_pd["failure_next_24h"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (700800, 8)
y_train shape: (700800,)
X_test shape: (175200, 8)
y_test shape: (175200,)


In [0]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [0]:
y_test_probability = model.predict_proba(X_test)[:, 1]
y_test_prediction = model.predict(X_test)

print("Probability predictions created successfully.")

print("First 10 failure probabilities:")
print(y_test_probability[:10])

print("\nFirst 10 predicted classes:")
print(y_test_prediction[:10])

Probability predictions created successfully.
First 10 failure probabilities:
[0.59747694 0.16081569 0.84919253 0.51947351 0.24344599 0.10406615
 0.43785639 0.33423095 0.84350368 0.40857069]

First 10 predicted classes:
[1 0 1 1 0 0 0 0 1 0]


In [0]:
y_test_probability = model.predict_proba(X_test)[:, 1]
y_test_prediction = model.predict(X_test)

print("Probability predictions created successfully.")

print("First 10 failure probabilities:")
print(y_test_probability[:10])

print("\nFirst 10 predicted classes:")
print(y_test_prediction[:10])

Probability predictions created successfully.
First 10 failure probabilities:
[0.59747694 0.16081569 0.84919253 0.51947351 0.24344599 0.10406615
 0.43785639 0.33423095 0.84350368 0.40857069]

First 10 predicted classes:
[1 0 1 1 0 0 0 0 1 0]


In [0]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

accuracy = accuracy_score(y_test, y_test_prediction)
precision = precision_score(y_test, y_test_prediction)
recall = recall_score(y_test, y_test_prediction)
roc_auc = roc_auc_score(y_test, y_test_probability)

print("Model Evaluation")
print("------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("ROC-AUC  :", roc_auc)

Model Evaluation
------------------------
Accuracy : 0.7042694063926941
Precision: 0.037479806138933765
Recall   : 0.6278255332696594
ROC-AUC  : 0.7720850528458789


In [0]:
latest_machine_features = (
    model_features
    .orderBy("machineID", col("datetime").desc())
    .dropDuplicates(["machineID"])
)

print("Latest machine records:", latest_machine_features.count())

latest_machine_features.select(
    "machineID",
    "datetime",
    "avg_volt_7d",
    "std_volt_7d",
    "avg_rotate_7d",
    "std_rotate_7d",
    "avg_pressure_7d",
    "std_pressure_7d",
    "avg_vibration_7d",
    "std_vibration_7d"
).orderBy("machineID").show(10, truncate=False)

Latest machine records: 100
+---------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+
|machineID|datetime           |avg_volt_7d       |std_volt_7d       |avg_rotate_7d     |std_rotate_7d     |avg_pressure_7d   |std_pressure_7d   |avg_vibration_7d  |std_vibration_7d |
+---------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+
|1        |2016-01-01 06:00:00|169.90090591517122|14.90656351617626 |449.4594652801717 |51.01208031751231 |99.42346514488707 |10.202380295571702|39.96549564271103 |5.399959890355803|
|2        |2016-01-01 06:00:00|171.05133052556732|15.010872122290266|425.1606009200436 |62.97712875669987 |98.96483039092847 |9.837759831997555 |39.88934687590294 |5.154390945880206|
|3        |2016-01-01 06:00:00|170.36127773105096|16.1629

In [0]:
latest_pd = latest_machine_features.toPandas()

X_latest = latest_pd[feature_columns]

latest_pd["failure_probability_24h"] = model.predict_proba(X_latest)[:, 1]

latest_pd["failure_risk_percent"] = (
    latest_pd["failure_probability_24h"] * 100
)

print("Failure risk calculated for:", len(latest_pd), "machines")

print("\nTop 10 machines by 24-hour failure probability:")

print(
    latest_pd[
        ["machineID", "datetime", "failure_probability_24h", "failure_risk_percent"]
    ]
    .sort_values("failure_probability_24h", ascending=False)
    .head(10)
    .to_string(index=False)
)

Failure risk calculated for: 100 machines

Top 10 machines by 24-hour failure probability:
 machineID            datetime  failure_probability_24h  failure_risk_percent
        15 2016-01-01 06:00:00                 0.985919             98.591871
        28 2016-01-01 06:00:00                 0.971012             97.101176
        44 2016-01-01 06:00:00                 0.919353             91.935279
        94 2016-01-01 06:00:00                 0.895704             89.570404
        63 2016-01-01 06:00:00                 0.887536             88.753607
        83 2016-01-01 06:00:00                 0.878575             87.857547
        85 2016-01-01 06:00:00                 0.874527             87.452675
        17 2016-01-01 06:00:00                 0.859847             85.984716
        88 2016-01-01 06:00:00                 0.827615             82.761540
        50 2016-01-01 06:00:00                 0.819795             81.979534


In [0]:
from pyspark.sql import functions as F

gold_q6_pd = latest_pd[
    [
        "machineID",
        "datetime",
        "avg_volt_7d",
        "std_volt_7d",
        "avg_rotate_7d",
        "std_rotate_7d",
        "avg_pressure_7d",
        "std_pressure_7d",
        "avg_vibration_7d",
        "std_vibration_7d",
        "failure_probability_24h",
        "failure_risk_percent"
    ]
]

gold_q6 = spark.createDataFrame(gold_q6_pd)

print("Gold Q6 data created.")
print("Records:", gold_q6.count())

gold_q6.printSchema()

gold_q6.orderBy(
    F.col("failure_probability_24h").desc()
).show(10, truncate=False)

Gold Q6 data created.
Records: 100
root
 |-- machineID: integer (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- avg_volt_7d: double (nullable = true)
 |-- std_volt_7d: double (nullable = true)
 |-- avg_rotate_7d: double (nullable = true)
 |-- std_rotate_7d: double (nullable = true)
 |-- avg_pressure_7d: double (nullable = true)
 |-- std_pressure_7d: double (nullable = true)
 |-- avg_vibration_7d: double (nullable = true)
 |-- std_vibration_7d: double (nullable = true)
 |-- failure_probability_24h: double (nullable = true)
 |-- failure_risk_percent: double (nullable = true)

+---------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------------+--------------------+
|machineID|datetime           |avg_volt_7d       |std_volt_7d       |avg_rotate_7d     |std_rotate_7d     |avg_pressure_7d   |std_pressure_7d   |avg_vibration_7d |std_vibrat

In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.gold_q6_failure_risk"

if spark.catalog.tableExists(target_table):

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("t")
        .merge(
            gold_q6.alias("s"),
            "t.machineID = s.machineID AND t.datetime = s.datetime"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Gold Q6 table updated using MERGE.")

else:

    (
        gold_q6.write
        .format("delta")
        .saveAsTable(target_table)
    )

    print("Gold Q6 table created.")

print("\nGold Q6 Schema:")
spark.table(target_table).printSchema()

print("\nGold Q6 Record Count:")
print(spark.table(target_table).count())

Gold Q6 table created.

Gold Q6 Schema:
root
 |-- machineID: integer (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- avg_volt_7d: double (nullable = true)
 |-- std_volt_7d: double (nullable = true)
 |-- avg_rotate_7d: double (nullable = true)
 |-- std_rotate_7d: double (nullable = true)
 |-- avg_pressure_7d: double (nullable = true)
 |-- std_pressure_7d: double (nullable = true)
 |-- avg_vibration_7d: double (nullable = true)
 |-- std_vibration_7d: double (nullable = true)
 |-- failure_probability_24h: double (nullable = true)
 |-- failure_risk_percent: double (nullable = true)


Gold Q6 Record Count:
100


In [0]:
display(
    gold_q6
    .orderBy(F.col("failure_probability_24h").desc())
    .limit(10)
    .select(
        "machineID",
        "failure_risk_percent"
    )
)

machineID,failure_risk_percent
15,98.5918710024111
28,97.10117637239894
44,91.93527939420358
94,89.57040443765206
63,88.75360677114166
83,87.85754727871199
85,87.45267541418173
17,85.9847157305816
88,82.76153953755471
50,81.9795336927456


Databricks visualization. Run in Databricks to view.